<a href="https://colab.research.google.com/github/Zikirullo/clear-vision/blob/develop/notebooks/02_merge_datasets.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 02 · Merge datasets

Combines the raw datasets on Drive (`MyDrive/clear_vision/datasets/raw/`) and the new
sources downloaded in section 2 (`/content/raw/`) into **one file on Drive**:

```
MyDrive/clear_vision/datasets/merged.zip          YOLO dataset (ZIP_STORED)
MyDrive/clear_vision/datasets/merge_report.json   boxes per class per split, per source
MyDrive/clear_vision/datasets/review/             sample images to review
MyDrive/clear_vision/datasets/merged.done         written only after a verified upload
```

- **Why a zip:** an earlier merged *folder* didn't fit on Drive, and the background upload
  silently lost most of its files. One file can be checked after upload.
- **What the merge does:**
  - Images over `MAX_SIDE` px are resized (JPEG q90).
  - Everything is built on the Colab disk.
  - It checks free Drive space before writing, flushes Drive, and verifies the zip's
    size and CRCs.
- **Mappings:** the class list is `configs/classes.yaml`, and each source has a mapping
  file `configs/mappings/<name>.yaml`. The merge fails if a source class isn't mapped.
- **Append** (`APPEND = True`) adds `NEW_SOURCES` to the existing zip instead of
  rebuilding it. It also re-maps merged sources whose mapping file changed, for example
  `pedestrian-crossing: drop` → `crosswalk` in `rf_crosswalk.yaml`.
- **v2 sources (2026-11-24):** four Roboflow datasets for the weakest hazards (rf_freesight,
  rf_neytra, rf_safewalk, rf_drone_obstacles) and ~3,000 indoor **negatives** (COCO images
  with no labels), appended with `APPEND = True`:
  - **Focus only** (`FOCUS_ONLY`, per source): keeps images with curb, stairs, pothole, pole
    or manhole, plus at most 20% other images. Appended images are resized to 800 px.
  - **Leakage guard:** Roboflow datasets often remix the same photos. A perceptual hash
    (pHash) of each new image is compared with every merged image. One within 6 bits of a
    val/test image is dropped; one near a train image goes to train. Printed per source.
  - **External test:** all of **rf_safewalk** goes to its own split `external_test` (never
    train/val/test), so notebook 10 can score the model on a source it has never seen.
  - The class table and the zip size are printed before the upload; if the Drive space
    check fails, nothing is uploaded.
- **Training** notebooks load the dataset with `src/colab_data.py` `prepare_dataset()`,
  never from Drive directly (see section 4).
- No GPU is needed. Run all cells.

In [ ]:
# Settings
REPO_URL = "https://github.com/Zikirullo/clear-vision.git"
BRANCH = "develop"
DRIVE_ROOT = "/content/drive/MyDrive/clear_vision"
MERGED_ZIP = f"{DRIVE_ROOT}/datasets/merged.zip"
WORK = "/content/merged_tmp"  # local build folder (Colab disk)
SEED = 42
MAX_SIDE = 1280  # full merge: images are resized so the longer side is at most this (JPEG q90)
FORCE = False  # True rebuilds merged.zip even if it is already done

# New Roboflow sources (configs/datasets.yaml), downloaded to LOCAL_RAW in section 2.
# The full merge and the append both read LOCAL_RAW together with the raw folder on Drive.
DOWNLOAD_NEW = ["rf_freesight", "rf_neytra", "rf_safewalk", "rf_drone_obstacles"]  # [] = skip
LOCAL_RAW = "/content/raw"
NEGATIVES = 3000  # indoor background images without labels (COCO) -> source "negatives"; 0 = skip
# Screen the negatives with budget_v1: an image where it sees stairs, curb, pole, pothole, manhole,
# bollard, barrier or traffic_cone is dropped and replaced by another COCO indoor image.
NEGATIVES_SCREEN_MODEL = f"{DRIVE_ROOT}/exports/budget_v1/best.pt"
NEGATIVES_SCREEN_CONF = 0.25
NEGATIVES_SCREEN_IMGSZ = 512

# Append mode: add sources to the existing merged.zip instead of rebuilding it.
APPEND = True
NEW_SOURCES = ["rf_freesight", "rf_neytra", "rf_safewalk", "rf_drone_obstacles", "negatives"]
APPEND_MAX_SIDE = 800  # appended images: longer side at most this (Drive has ~9 GB free)
# Focus only, per source: keep images with curb, stairs, pothole, pole or manhole, plus other
# images up to OTHER_SHARE of the kept ones. Never for "negatives" (they have no boxes).
FOCUS_ONLY = {"rf_freesight": True, "rf_neytra": True, "rf_safewalk": True, "rf_drone_obstacles": True}
OTHER_SHARE = 0.2
PHASH_DISTANCE = 6  # leakage guard: images whose pHashes differ in <= this many bits are near-duplicates

## 1. Setup

In [ ]:
# 1. Setup: Drive, repo, packages
import logging
import subprocess
import sys
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

REPO_DIR = Path("/content/clear-vision")
if (REPO_DIR / ".git").exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "-q", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "-q", "-B", BRANCH, f"origin/{BRANCH}"], check=True)
else:
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO_URL, str(REPO_DIR)], check=True)
commit = subprocess.run(["git", "-C", str(REPO_DIR), "log", "--oneline", "-1"], capture_output=True, text=True).stdout
print(f"Repo at {REPO_DIR}: {commit.strip()}")

%pip install -q ultralytics roboflow imagehash
if NEGATIVES and "negatives" in NEW_SOURCES:  # FiftyOne's COCO zoo, for the indoor negatives
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "fiftyone"], check=True)

sys.path.insert(0, str(REPO_DIR))
sys.path.insert(0, str(REPO_DIR / "scripts"))
import download_datasets  # noqa: E402
import merge_datasets  # noqa: E402
from src.colab_data import prepare_dataset  # noqa: E402

logging.basicConfig(level=logging.INFO, format="%(message)s", force=True)

RAW = Path(DRIVE_ROOT) / "datasets" / "raw"
MERGED_ZIP = Path(MERGED_ZIP)
print(f"raw on Drive: {RAW}")
print(f"merged zip:   {MERGED_ZIP}")
print(f"Drive free:   {merge_datasets.gb(merge_datasets.free_space(MERGED_ZIP.parent))}")

## 2. Download new sources

Downloads `DOWNLOAD_NEW` (Roboflow, listed in `configs/datasets.yaml`) to `LOCAL_RAW` with the same code as notebook 01. Finished downloads (with `.done`) are skipped. If a version fails, the newest version is used (printed as `USED LATEST VERSION`).

**Review the class check:** for each dataset it prints the **real class names** and where each one goes (`-> curb`, `-> drop`), unmapped classes, and `CHECK:` lines for dropped names that look like ours (e.g. `Curbs`). Fix `configs/mappings/<name>.yaml`, push, and re-run if something is missed.

Then it downloads `NEGATIVES` indoor COCO-2017 images (chair, couch, tv, bed, ... and none of person, car, dog, bench, traffic light, ...) to `LOCAL_RAW/negatives/` with empty label files. They are **screened with budget_v1** (conf 0.25, imgsz 512):

- An image where it detects stairs, curb, pole, pothole, manhole, bollard, barrier or traffic_cone may really contain one, so it is dropped.
- More COCO images are taken until about 3,000 remain, or COCO runs out, which is printed.
- It prints how many were dropped, per class, and shows 8 dropped examples (red = what budget_v1 saw).

In [ ]:
# 2. Download new sources to this runtime's disk (LOCAL_RAW)
# Each Colab notebook has its own runtime, so downloads from notebook 01 are not here.
if DOWNLOAD_NEW:
    new_datasets = download_datasets.select_datasets(
        download_datasets.load_roboflow_datasets(REPO_DIR / "configs" / "datasets.yaml"), DOWNLOAD_NEW
    )
    failures = download_datasets.download_roboflow_datasets(
        new_datasets, Path(LOCAL_RAW), download_datasets.colab_secret("ROBOFLOW_API_KEY")
    )
    download_datasets.check_mappings(
        Path(LOCAL_RAW), DOWNLOAD_NEW, download_datasets.mappings_from_dir(REPO_DIR / "configs" / "mappings")
    )
    if failures:
        raise RuntimeError(f"Downloads failed, fix them before merging: {failures}")
else:
    print("DOWNLOAD_NEW is empty: nothing to download")

if NEGATIVES and "negatives" in NEW_SOURCES:
    from collections import Counter

    import matplotlib.pyplot as plt

    screen_model = Path(NEGATIVES_SCREEN_MODEL)
    if not screen_model.exists():
        raise RuntimeError(f"{screen_model} not found: the negatives are screened with it (NEGATIVES_SCREEN_MODEL).")
    screen = download_datasets.make_yolo_screen(screen_model, NEGATIVES_SCREEN_IMGSZ, NEGATIVES_SCREEN_CONF)
    NEG = download_datasets.download_negatives(
        Path(LOCAL_RAW), NEGATIVES, screen=screen, conf=NEGATIVES_SCREEN_CONF,
        screen_name=f"{screen_model.parent.name}/{screen_model.name}",
    )
    dropped = NEG["dropped"]
    if NEG["kept"] or dropped:
        by_class = Counter(c for hits in dropped.values() for c in {h[0] for h in hits})
        print(f"negatives: {len(NEG['kept'])} kept, {len(dropped)} dropped by budget_v1 "
              f"(images per class: {dict(by_class.most_common())})")
        if not NEG["enough"]:
            print(f"WARNING: only {len(NEG['kept'])} of {NEGATIVES} wanted: COCO has no more indoor images "
                  "without our classes that pass the screen.")
        examples = sorted(dropped)[:: max(1, len(dropped) // 8)][:8]
        if examples:
            fig, axes = plt.subplots(2, 4, figsize=(16, 7))
            for ax in axes.flat:
                ax.axis("off")
            for ax, path in zip(axes.flat, examples):
                ax.imshow(download_datasets.draw_hits(path, dropped[path]))
                ax.set_title(", ".join(sorted({h[0] for h in dropped[path]})), fontsize=9)
            fig.suptitle("Dropped negatives: what budget_v1 saw (red)")
            plt.tight_layout()
            plt.show()
    else:
        print("negatives: already downloaded and screened")

## 3. Merge or append

In append mode it also prints, per source, how many images focus-only left out and how many near-duplicates were dropped or put in train (`focus-`, `near-dup-`, `->train` columns), and the class table with an `external` column for rf_safewalk.

Prints the mappings, the raw sizes on Drive, the image size before/after resize, the zip size, the space check, the verification and the Drive space left. It stops with a message (and no `merged.done`) if anything doesn't fit or doesn't verify.

In [ ]:
# 3. Merge or append -> merged.zip on Drive (mappings, progress, sizes, class table)
import json

CLASSES_PATH = REPO_DIR / "configs" / "classes.yaml"
MAPPINGS_DIR = REPO_DIR / "configs" / "mappings"
raw_dirs = [Path(LOCAL_RAW), RAW] if Path(LOCAL_RAW).is_dir() else [RAW]  # local copy wins

if APPEND:
    report = merge_datasets.append(
        raw=raw_dirs,
        zip_path=MERGED_ZIP,
        new_sources=NEW_SOURCES,
        work=WORK,
        classes_path=CLASSES_PATH,
        mappings_dir=MAPPINGS_DIR,
        seed=SEED,
        max_side=APPEND_MAX_SIDE,
        focus_only=[name for name, on in FOCUS_ONLY.items() if on and name in NEW_SOURCES],
        other_share=OTHER_SHARE,
        phash_distance=PHASH_DISTANCE,
    )
else:
    report = merge_datasets.merge(
        raw=raw_dirs,
        zip_path=MERGED_ZIP,
        work=WORK,
        classes_path=CLASSES_PATH,
        mappings_dir=MAPPINGS_DIR,
        seed=SEED,
        max_side=MAX_SIDE,
        force=FORCE,
    )
if report is None:  # nothing to do: show the saved report
    report = json.loads(MERGED_ZIP.with_name("merge_report.json").read_text())
    merge_datasets.log_summary(report)

## 4. Load and check (as training will)

In [ ]:
# 4. Load the dataset the way training will: copy merged.zip to the Colab disk, unzip,
#    compare image counts with merge_report.json, run Ultralytics' check_det_dataset.
#    Later notebooks (report, training) use exactly this line.
DATA_YAML = prepare_dataset(MERGED_ZIP)
print(f"OK: {DATA_YAML}")


## 5. Review

Sample images of review classes (red boxes; green = kept boxes):

- **`Light` (v2 sources):** dropped for now. If the red boxes are **traffic lights**, change `Light: drop` to `Light: traffic_light` in that source's mapping file, push, and run this notebook again with `APPEND = True`. Only that source's labels are re-mapped. If they are lamps or street lights, leave it.
- **`pedestrian-crossing` (rf_crosswalk):** decide whether it maps to `crosswalk` the same way.

In [ ]:
# 5. Review samples: pedestrian-crossing boxes (red) that are dropped for now
import matplotlib.image as mpimg
import matplotlib.pyplot as plt

for folder in sorted((MERGED_ZIP.parent / "review").glob("*")):
    files = sorted(folder.glob("*"))[:16]
    print(f"{folder.name}: {len(files)} images (red = review class, green = kept boxes)")
    if not files:
        continue
    cols = 4
    rows = (len(files) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(16, 4 * rows))
    for ax in axes.flat:
        ax.axis("off")
    for ax, path in zip(axes.flat, files):
        ax.imshow(mpimg.imread(path))
        ax.set_title(path.name[:40], fontsize=8)
    plt.tight_layout()
    plt.show()
